In [ ]:
from viafoundry.client import ViaFoundryClient
import json


In [ ]:
# Make sure to authenticate it
#client = ViaFoundryClient("~/.viaprod",  enable_session_history=True)
client = ViaFoundryClient("/Users/alper/.vialocal")


In [ ]:
report_data = client.reports.fetch_report_data(report_id="1")

process_names = client.reports.get_process_names(report_data)
print("Processes:", process_names)
filenames = client.reports.get_file_names(report_data, "RSEM_module")
print(filenames)
# We will use file_path to load the data


In [ ]:
rsem_data = client.reports.load_file(report_data, "rsem_summary/genes_expression_expected_count.tsv")
# To download the file you can use the line below.
# client.reports.download_file(report_data, "rsem_summary/genes_expression_expected_count.tsv")

print(rsem_data)

In [ ]:
directories = client.reports.get_report_dirs(report_id="1")
print("Directories:", directories)

In [ ]:
# prompt: Write a script to Create a scatter plot using rsem_data by getting the average of LPS vs ctrl columns in ligth grey and log Scale. The column prefixes are LPS and ctrl. Make sure to complete full column name yourself to get average for each condition. And save the file as plot.png

import pandas as pd
import matplotlib.pyplot as plt

# Assuming rsem_data is a pandas DataFrame
# Calculate the average of LPS and ctrl columns
lps_cols = [col for col in rsem_data.columns if col.startswith('control')]
ctrl_cols = [col for col in rsem_data.columns if col.startswith('exper')]

if not lps_cols or not ctrl_cols:
    print("No LPS or ctrl columns found in the data.")
else:
    rsem_data['LPS_avg'] = rsem_data[lps_cols].mean(axis=1)
    rsem_data['ctrl_avg'] = rsem_data[ctrl_cols].mean(axis=1)

    # Create the scatter plot
    plt.figure(figsize=(10, 6))
    plt.scatter(rsem_data['ctrl_avg'], rsem_data['LPS_avg'], color='lightgrey')
    plt.xlabel('Average ctrl Expression', fontsize=12)
    plt.ylabel('Average LPS Expression', fontsize=12)
    plt.title('LPS vs ctrl Expression', fontsize=14)
    plt.xscale('log')
    plt.yscale('log')
    plt.savefig('plot.png')
    plt.show()

In [ ]:
response = client.reports.upload_report_file(
    report_id="1",
    local_file_path="plot.png",
    dir="summary"
)

In [ ]:
client.reports.enable_session_history=True
response = client.reports.upload_session_history(
    report_id="1",
    dir="summary"
)

In [ ]:
%history

In [ ]:
#Fetcht report 1
#list_files for report 1
#load rRNA file
#Create a percentage barplot for rRNA content


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('/Users/alper/Desktop/download/rRNA.counts.tsv', sep='\t')

# List of sample columns
samples = ['control_rep1', 'control_rep2', 'control_rep3', 'exper_rep1', 'exper_rep2', 'exper_rep3']

# Calculate total rRNA reads per sample
totals = df[samples].sum()

# Calculate percentages
percent_df = df.copy()
for sample in samples:
    percent_df[sample] = percent_df[sample] / totals[sample] * 100

# Melt dataframe for seaborn-friendly format
df_melted = percent_df.melt(id_vars=['id'], value_vars=samples, var_name='Sample', value_name='Percent')

# Plot barplot
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.barplot(data=df_melted, x='Sample', y='Percent', hue='id')
plt.ylabel("rRNA content (%)")
plt.title("rRNA Content Percentage per Sample")
plt.xticks(rotation=45)
plt.legend(title='rRNA Type')
plt.tight_layout()
plt.show()